# 数理変換タクティクス AI トレーニング
## ハイブリッドEmbedding版（数学特徴 + 学習特徴）

### モデルの構造
```
カード整数
  │
  ├─ 数学Embedding（固定・非学習）
  │    素因数, 約数数, 桁の和, 素数フラグ → 9次元
  │
  └─ 学習Embedding（学習・更新される）
       ゲーム戦略を自動発見 → 16次元
  │
  結合 → 25次元/枚
  │
  GlobalMaxPool（手札全体を25次元に圧縮）
  │
  ×2（自分・相手）＋スカラー5次元 = 55次元
  │
  Dense → 7クラス（操作選択）
```

### 誤差逆伝播の流れ
```
予測結果 → 損失計算 → 誤差を逆向きに伝播

Dense層        ← 誤差あり、重み更新
GlobalMaxPool  ← 誤差あり（勾配通過）
学習Embedding  ← 誤差あり、重み更新  ★
数学Embedding  ← 誤差通過するが重み更新しない（固定）
```
数学的に正しい値は最初から与えるので、
学習Embeddingは「どの数字がゲーム上強いか」だけに集中できる。

In [ ]:
!pip install tensorflowjs scikit-learn -q
print('完了！')

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflowjs as tfjs
import random
from sklearn.model_selection import train_test_split

MAX_HAND    = 20
EMBED_DIM   = 16
NUM_ACTIONS = 7
MAX_CARD    = 150
PRIMES      = [2, 3, 5, 7, 11, 13]
MATH_DIM    = len(PRIMES) + 3  # 9次元

DIM_NAMES = [
    '2の指数', '3の指数', '5の指数', '7の指数', '11の指数', '13の指数',
    '大素数フラグ', '約数の個数', '桁の和'
]

print(f'TensorFlow: {tf.__version__}')
print(f'数学特徴: {MATH_DIM}次元 → {DIM_NAMES}')

## 数学特徴テーブルの構築

In [ ]:
def card_math_features(n):
    if n == 0:
        return [0.0] * MATH_DIM
    orig = n
    factors = []
    for p in PRIMES:
        count = 0
        while n % p == 0:
            n //= p; count += 1
        factors.append(min(count, 4) / 4.0)
    has_large_prime = 1.0 if n > 1 else 0.0
    num_divs = sum(1 for d in range(1, orig+1) if orig % d == 0)
    ds = sum(int(d) for d in str(orig))
    return factors + [has_large_prime, min(num_divs,20)/20.0, min(ds,30)/30.0]

FACTOR_TABLE = np.array(
    [card_math_features(i) for i in range(MAX_CARD + 1)], dtype=np.float32
)

print(f'テーブル shape: {FACTOR_TABLE.shape}')
for n in [6, 7, 12, 17, 30]:
    f = card_math_features(n)
    print(f'  {n:3d}: {[round(x,2) for x in f]}')

## ゲームシミュレーター

In [ ]:
def gcd(a, b):
    while b: a, b = b, a % b
    return a

def digit_sum(n):
    return sum(int(d) for d in str(n))

class MathCardGame:
    def __init__(self, config=None):
        self.config = config or {
            'initHandCount': 5, 'initMaxNum': 10,
            'drawCount': 2, 'drawMaxNum': 20,
            'handLimitNum': MAX_CARD, 'winScore': 100, 'maxTurns': 10,
        }

    def reset(self):
        cfg = self.config
        self.hands = {
            'P1': [random.randint(1, cfg['initMaxNum']) for _ in range(cfg['initHandCount'])],
            'P2': [random.randint(1, cfg['initMaxNum']) for _ in range(cfg['initHandCount'])],
        }
        self.scores = {'P1': 0, 'P2': 0}
        self.turn_count = 1
        self.current_player = 'P1'
        self.done = False; self.winner = None

    def opponent(self, pid): return 'P2' if pid == 'P1' else 'P1'

    def get_inputs(self, pid):
        my  = self.hands[pid]
        enm = self.hands[self.opponent(pid)]
        pad = lambda arr: (arr[:MAX_HAND] + [0]*MAX_HAND)[:MAX_HAND]
        scalars = [
            self.scores[pid] / 100.0,
            self.scores[self.opponent(pid)] / 100.0,
            self.turn_count / 10.0,
            len(my)  / MAX_HAND,
            len(enm) / MAX_HAND,
        ]
        return pad(my), pad(enm), scalars

    def best_attack(self, pid):
        enm = self.hands[self.opponent(pid)]
        best, bg = None, 0
        for i, a in enumerate(self.hands[pid]):
            if a == 1: continue
            g = sum(n for n in enm if n % a == 0)
            if g > bg: bg = g; best = (i, a, g)
        return best

    def best_add(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                r = h[i]+h[j]
                if r<=lim and r>bv: bv=r; best=(i,j,r)
        return best

    def best_sub(self, pid):
        h = self.hands[pid]; best, bv = None, 0
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                r = h[i]-h[j]
                if r>0 and r>bv: bv=r; best=(i,j,r)
        return best

    def best_divprod(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j or h[j]==0: continue
                r = (h[i]//h[j])*(h[i]%h[j])
                if r>0 and r<=lim and r>bv: bv=r; best=(i,j,r)
        return best

    def best_dsd(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i, num in enumerate(h):
            ds = digit_sum(num)
            if ds==0: continue
            q, r = divmod(num, ds)
            if q<=lim and q>bv: bv=q; best=(i,q,r)
        return best

    def best_gm(self, pid):
        h, lim = self.hands[pid], self.config['handLimitNum']
        best, bv = None, -1
        for i in range(len(h)):
            for j in range(len(h)):
                if i==j: continue
                g = gcd(h[i], h[j])
                if g<=1: continue
                r = h[i]*g
                if r<=lim and r>bv: bv=r; best=(i,j,r)
        return best

    def get_optimal_action(self, pid):
        atk = self.best_attack(pid); add = self.best_add(pid)
        sub = self.best_sub(pid);   dp  = self.best_divprod(pid)
        dsd = self.best_dsd(pid);   gm  = self.best_gm(pid)
        ms = self.scores[pid]; win = self.config['winScore']
        if atk and self.turn_count>1 and ms+atk[2]>=win: return 1
        if gm  and gm[2]>=30:  return 6
        if atk and self.turn_count>1 and atk[2]>=10: return 1
        if add and add[2]>=20: return 2
        if dp  and dp[2]>=15:  return 4
        if atk and self.turn_count>1: return 1
        if gm  and gm[2]>=10: return 6
        if dsd and dsd[1]>=5:  return 5
        if add: return 2
        if dp:  return 4
        if sub: return 3
        return 0

    def execute_action(self, pid, action):
        h = self.hands[pid]; eid = self.opponent(pid)
        if action == 1:
            atk = self.best_attack(pid)
            if atk:
                i,v,gain = atk
                self.hands[eid] = [n for n in self.hands[eid] if n%v!=0]
                h.pop(i); self.scores[pid] += gain
                if self.scores[pid] >= self.config['winScore']:
                    self.done = True; self.winner = pid
        elif action == 2:
            a = self.best_add(pid)
            if a: i,j,r=a; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)
        elif action == 3:
            s = self.best_sub(pid)
            if s: i,j,r=s; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)
        elif action == 4:
            d = self.best_divprod(pid)
            if d: i,j,r=d; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)
        elif action == 5:
            d = self.best_dsd(pid)
            if d: idx,q,r=d; h.pop(idx); h.append(q); (h.append(r) if r>0 else None)
        elif action == 6:
            g = self.best_gm(pid)
            if g: i,j,r=g; h=[c for k,c in enumerate(h) if k!=i and k!=j]; h.append(r)
        self.hands[pid] = h
        self.current_player = eid
        if self.current_player == 'P1':
            self.turn_count += 1
            if self.turn_count > self.config['maxTurns']:
                self.done = True
                self.winner = max(self.scores, key=lambda p: self.scores[p])

print('準備完了！')

## トレーニングデータ生成

In [ ]:
NUM_EPISODES = 30000
my_hands_data, enemy_hands_data, scalars_data, y_data = [], [], [], []
game = MathCardGame()

for ep in range(NUM_EPISODES):
    game.reset()
    while not game.done:
        pid = game.current_player
        my_h, enm_h, sc = game.get_inputs(pid)
        action = game.get_optimal_action(pid)
        my_hands_data.append(my_h); enemy_hands_data.append(enm_h)
        scalars_data.append(sc);   y_data.append(action)
        game.execute_action(pid, action)
    if (ep+1) % 10000 == 0:
        print(f'{ep+1}/{NUM_EPISODES} 完了... データ数: {len(y_data)}')

my_hands_data    = np.array(my_hands_data,    dtype=np.int32)
enemy_hands_data = np.array(enemy_hands_data, dtype=np.int32)
scalars_data     = np.array(scalars_data,     dtype=np.float32)
y_data           = np.array(y_data,           dtype=np.int32)
print(f'総データ数: {len(y_data)}')

## モデル構築

In [ ]:
from tensorflow.keras.utils import to_categorical

idx = np.arange(len(y_data))
train_idx, val_idx = train_test_split(idx, test_size=0.1, random_state=42)

def make_dataset(indices, batch_size=256, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((
        {'my_hand': my_hands_data[indices],
         'enemy_hand': enemy_hands_data[indices],
         'scalars': scalars_data[indices]},
        to_categorical(y_data[indices], NUM_ACTIONS)
    ))
    if shuffle: ds = ds.shuffle(10000)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_idx)
val_ds   = make_dataset(val_idx, shuffle=False)

my_hand_in    = tf.keras.Input(shape=(MAX_HAND,), dtype='int32',   name='my_hand')
enemy_hand_in = tf.keras.Input(shape=(MAX_HAND,), dtype='int32',   name='enemy_hand')
scalar_in     = tf.keras.Input(shape=(5,),        dtype='float32', name='scalars')

math_emb = tf.keras.layers.Embedding(
    MAX_CARD+1, MATH_DIM, weights=[FACTOR_TABLE],
    trainable=False, mask_zero=True, name='math_emb'
)
learned_emb = tf.keras.layers.Embedding(
    MAX_CARD+1, EMBED_DIM, mask_zero=True, name='learned_emb'
)

def encode_hand(hand_input):
    combined = tf.keras.layers.Concatenate(axis=-1)([
        math_emb(hand_input), learned_emb(hand_input)
    ])
    return tf.keras.layers.GlobalMaxPooling1D()(combined)

x = tf.keras.layers.Concatenate(name='combined')([
    encode_hand(my_hand_in), encode_hand(enemy_hand_in), scalar_in
])
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
out = tf.keras.layers.Dense(NUM_ACTIONS, activation='softmax', name='action')(x)

model = tf.keras.Model(inputs=[my_hand_in, enemy_hand_in, scalar_in], outputs=out)
model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss='categorical_crossentropy', metrics=['accuracy'])

total     = model.count_params()
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f'総パラメータ: {total:,}  (学習: {trainable:,}  固定: {total-trainable:,})')

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
]
history = model.fit(train_ds, validation_data=val_ds,
                    epochs=60, callbacks=callbacks, verbose=1)
print(f'最高検証精度: {max(history.history["val_accuracy"])*100:.1f}%')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in zip(axes, ['accuracy','loss'], ['精度','損失']):
    ax.plot(history.history[key], label='訓練')
    ax.plot(history.history[f'val_{key}'], label='検証')
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

## 数学特徴の次元ごとの重要度（Permutation Importance）

### 考え方
```
ベースライン精度: 85%

次元1（2の指数）をランダムに入れ替え → 精度 72% → 低下 13% → 重要！
次元9（桁の和）をランダムに入れ替え → 精度 84% → 低下  1% → それほど重要でない
```

「壊すと困る」次元 = 重要、という発想。

In [ ]:
# ベースライン精度
base_loss, base_acc = model.evaluate(val_ds, verbose=0)
print(f'ベースライン精度: {base_acc*100:.2f}%')
print()

importances = []

for d in range(MATH_DIM):
    # 次元dだけシャッフルしたテーブルを作成
    shuffled = FACTOR_TABLE.copy()
    shuffled[:, d] = np.random.permutation(shuffled[:, d])

    # 数学Embeddingの重みを一時的に置き換え
    model.get_layer('math_emb').set_weights([shuffled])

    # 精度を測定
    _, acc = model.evaluate(val_ds, verbose=0)
    drop = base_acc - acc
    importances.append(drop)

    # 元に戻す
    model.get_layer('math_emb').set_weights([FACTOR_TABLE])

    print(f'次元{d+1:2d} [{DIM_NAMES[d]:12s}]: 精度低下 {drop*100:+.2f}%  {"★重要" if drop > 0.02 else ""}')

# グラフ表示
plt.figure(figsize=(10, 4))
colors = ['crimson' if v > 0.02 else 'steelblue' for v in importances]
bars = plt.barh(DIM_NAMES, [v*100 for v in importances], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('精度の低下 (%) ← 大きいほど重要')
plt.title('数学特徴の次元ごとの重要度（赤=重要）')
plt.tight_layout()
plt.show()

most_important = DIM_NAMES[np.argmax(importances)]
print(f'\n最も重要な次元: 【{most_important}】')

## TensorFlow.js 形式でエクスポート

In [ ]:
import os, shutil
OUT = 'tfjs_model'
if os.path.exists(OUT): shutil.rmtree(OUT)
tfjs.converters.save_keras_model(model, OUT)
print('エクスポート完了！')
for f in os.listdir(OUT):
    print(f'  {f} ({os.path.getsize(os.path.join(OUT,f))/1024:.1f} KB)')
shutil.make_archive('tfjs_model', 'zip', 'tfjs_model')
from google.colab import files
files.download('tfjs_model.zip')
print('ダウンロード開始！')

## 動作テスト

In [ ]:
names = ['パス','攻撃','足し算','引き算','商×余','桁和分裂','GCD掛け']
tests = [
    {'p1':[6,3,10,7,2],  'p2':[12,18,5,9,4],          'desc':'攻撃チャンスあり'},
    {'p1':[5,8,3,2,7],   'p2':[1,1,1,1,1],             'desc':'合成が有利'},
    {'p1':[6,9,4,3,2],   'p2':[8,10,6,4,3],            'desc':'GCD有効'},
    {'p1':[36,12,24,48], 'p2':[7,11,13,17,19,23],      'desc':'相手が全員素数'},
]
for t in tests:
    g = MathCardGame(); g.reset()
    g.hands['P1'] = t['p1']; g.hands['P2'] = t['p2']
    g.turn_count = 3; g.scores = {'P1': 20, 'P2': 30}
    my_h, enm_h, sc = g.get_inputs('P1')
    pred = model.predict(
        {'my_hand': np.array([my_h]), 'enemy_hand': np.array([enm_h]),
         'scalars': np.array([sc])}, verbose=0
    )[0]
    chosen = int(np.argmax(pred)); optimal = g.get_optimal_action('P1')
    print(f'【{t["desc"]}】 AI: {names[chosen]} | 最適: {names[optimal]} | {"✅" if chosen==optimal else "❌"}')